# 베이스라인

## 전체

In [7]:
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


warnings.filterwarnings(
    "ignore",
    message=".*X does not have valid feature names.*LGBMClassifier.*",
    category=UserWarning,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_view_delete\Membership_v2.csv"

use_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "is_repurchase",
]


def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


def make_preprocessor():
    numeric_features = [
        "price",
        "max_screen",
        "is_promotion",
        "is_churn_prevented",
        "is_user_verified",
        "age",
    ]

    categorical_features = [
        "payment_device",
        "gender",
    ]

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )


def make_models(scale_pos_weight):
    return {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "SVM": SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoost": GradientBoostingClassifier(
            random_state=42,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=4,
            auto_class_weights="Balanced",
            random_state=42,
            verbose=0,
            allow_writing_files=False,
        ),
    }


df = pd.read_csv(file_path, usecols=use_cols).copy()

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

X = df[
    [
        "price",
        "max_screen",
        "is_promotion",
        "is_churn_prevented",
        "payment_device",
        "is_user_verified",
        "gender",
        "age",
    ]
].copy()

y = (df["is_repurchase_num"] == 0).astype(int)

print("분석 기준: 전체 데이터")
print("양성 클래스 기준: is_repurchase == 0")
print("타깃 분포")
print_target_distribution(y)

if y.nunique() < 2 or y.value_counts().min() < 2:
    print("학습 불가: 타깃 클래스가 부족합니다.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    print(f"train/test 분리 비율: train {len(X_train) / len(X) * 100:.1f}% / test {len(X_test) / len(X) * 100:.1f}%")
    print(f"train 데이터 수: {len(X_train)}")
    print(f"test 데이터 수: {len(X_test)}")

    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    models = make_models(scale_pos_weight)
    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", make_preprocessor()),
                ("model", model),
            ]
        )

        clf.fit(X_train, y_train)

        y_test_pred = clf.predict(X_test)

        y_train_proba = clf.predict_proba(X_train)[:, 1]
        y_test_proba = clf.predict_proba(X_test)[:, 1]

        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_test_proba)
        auc_gap = train_roc_auc - test_roc_auc

        results.append(
            {
                "model": model_name,
                "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
                "train_roc_auc": train_roc_auc,
                "test_roc_auc": test_roc_auc,
                "auc_gap": auc_gap,
                "is_overfit": auc_gap >= 0.05,
            }
        )

    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        [["f1_score", "train_roc_auc", "test_roc_auc", "auc_gap", "is_overfit"]]
        .round(4)
        .sort_values(
            by=["is_overfit", "test_roc_auc"],
            ascending=[True, False],
        )
    )

    print(results_df.to_string())

분석 기준: 전체 데이터
양성 클래스 기준: is_repurchase == 0
타깃 분포
                   count  percent
is_repurchase_num                
0                  16702    71.55
1                   6641    28.45
train/test 분리 비율: train 80.0% / test 20.0%
train 데이터 수: 18674
test 데이터 수: 4669
                    f1_score  train_roc_auc  test_roc_auc  auc_gap  is_overfit
model                                                                         
GradientBoost         0.0015         0.5929        0.5873   0.0056       False
CatBoost              0.4253         0.5999        0.5865   0.0135       False
XGBoost               0.4219         0.6116        0.5845   0.0270       False
LogisticRegression    0.4261         0.5711        0.5800  -0.0089       False
SVM                   0.4320         0.5846        0.5772   0.0074       False
LightGBM              0.4245         0.6304        0.5784   0.0520        True
RandomForest          0.4064         0.6495        0.5615   0.0880        True


## 비프로모션

In [5]:
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


warnings.filterwarnings(
    "ignore",
    message=".*X does not have valid feature names.*LGBMClassifier.*",
    category=UserWarning,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_view_delete\Membership_v2.csv"

use_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "is_repurchase",
]


def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


def make_preprocessor():
    numeric_features = [
        "price",
        "max_screen",
        "is_churn_prevented",
        "is_user_verified",
        "age",
    ]

    categorical_features = [
        "payment_device",
        "gender",
    ]

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )


def make_models(scale_pos_weight):
    return {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "SVM": SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoost": GradientBoostingClassifier(
            random_state=42,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=4,
            auto_class_weights="Balanced",
            random_state=42,
            verbose=0,
            allow_writing_files=False,
        ),
    }


df = pd.read_csv(file_path, usecols=use_cols).copy()

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

df_promotion_0 = df[df["is_promotion"] == 0].copy()

X = df_promotion_0[
    [
        "price",
        "max_screen",
        "is_churn_prevented",
        "payment_device",
        "is_user_verified",
        "gender",
        "age",
    ]
].copy()

y = (df_promotion_0["is_repurchase_num"] == 0).astype(int)

print("분석 기준: is_promotion == 0")
print("양성 클래스 기준: is_repurchase == 0")
print("타깃 분포")
print_target_distribution(y)

if y.nunique() < 2 or y.value_counts().min() < 2:
    print("학습 불가: 타깃 클래스가 부족합니다.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    print(f"train/test 분리 비율: train {len(X_train) / len(X) * 100:.1f}% / test {len(X_test) / len(X) * 100:.1f}%")
    print(f"train 데이터 수: {len(X_train)}")
    print(f"test 데이터 수: {len(X_test)}")

    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    models = make_models(scale_pos_weight)
    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", make_preprocessor()),
                ("model", model),
            ]
        )

        clf.fit(X_train, y_train)

        y_test_pred = clf.predict(X_test)

        y_train_proba = clf.predict_proba(X_train)[:, 1]
        y_test_proba = clf.predict_proba(X_test)[:, 1]

        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_test_proba)
        auc_gap = train_roc_auc - test_roc_auc

        results.append(
            {
                "model": model_name,
                "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
                "train_roc_auc": train_roc_auc,
                "test_roc_auc": test_roc_auc,
                "auc_gap": auc_gap,
                "is_overfit": auc_gap >= 0.05,
            }
        )

    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        [["f1_score", "train_roc_auc", "test_roc_auc", "auc_gap", "is_overfit"]]
        .round(4)
        .sort_values(
            by=["is_overfit", "test_roc_auc"],
            ascending=[True, False],
        )
    )

    print(results_df.to_string())

분석 기준: is_promotion == 0
양성 클래스 기준: is_repurchase == 0
타깃 분포
                   count  percent
is_repurchase_num                
0                   8642    75.89
1                   2746    24.11
train/test 분리 비율: train 80.0% / test 20.0%
train 데이터 수: 9110
test 데이터 수: 2278
                    f1_score  train_roc_auc  test_roc_auc  auc_gap  is_overfit
model                                                                         
GradientBoost         0.0036         0.5890        0.5413   0.0477       False
LogisticRegression    0.3365         0.5335        0.5302   0.0033       False
CatBoost              0.3653         0.6066        0.5440   0.0626        True
XGBoost               0.3391         0.6235        0.5368   0.0866        True
SVM                   0.3236         0.5800        0.5261   0.0539        True
LightGBM              0.3398         0.6466        0.5249   0.1217        True
RandomForest          0.3155         0.6670        0.5137   0.1533        True


## 프로모션

In [6]:
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


warnings.filterwarnings(
    "ignore",
    message=".*X does not have valid feature names.*LGBMClassifier.*",
    category=UserWarning,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_view_delete\Membership_v2.csv"

use_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "is_repurchase",
]


def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


def make_preprocessor():
    numeric_features = [
        "price",
        "max_screen",
        "is_churn_prevented",
        "is_user_verified",
        "age",
    ]

    categorical_features = [
        "payment_device",
        "gender",
    ]

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )


def make_models(scale_pos_weight):
    return {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "SVM": SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoost": GradientBoostingClassifier(
            random_state=42,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=4,
            auto_class_weights="Balanced",
            random_state=42,
            verbose=0,
            allow_writing_files=False,
        ),
    }


df = pd.read_csv(file_path, usecols=use_cols).copy()

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

df_promotion_1 = df[df["is_promotion"] == 1].copy()

X = df_promotion_1[
    [
        "price",
        "max_screen",
        "is_churn_prevented",
        "payment_device",
        "is_user_verified",
        "gender",
        "age",
    ]
].copy()

y = (df_promotion_1["is_repurchase_num"] == 0).astype(int)

print("분석 기준: is_promotion == 1")
print("양성 클래스 기준: is_repurchase == 0")
print("타깃 분포")
print_target_distribution(y)

if y.nunique() < 2 or y.value_counts().min() < 2:
    print("학습 불가: 타깃 클래스가 부족합니다.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    print(f"train/test 분리 비율: train {len(X_train) / len(X) * 100:.1f}% / test {len(X_test) / len(X) * 100:.1f}%")
    print(f"train 데이터 수: {len(X_train)}")
    print(f"test 데이터 수: {len(X_test)}")

    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    models = make_models(scale_pos_weight)
    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", make_preprocessor()),
                ("model", model),
            ]
        )

        clf.fit(X_train, y_train)

        y_test_pred = clf.predict(X_test)

        y_train_proba = clf.predict_proba(X_train)[:, 1]
        y_test_proba = clf.predict_proba(X_test)[:, 1]

        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_test_proba)
        auc_gap = train_roc_auc - test_roc_auc

        results.append(
            {
                "model": model_name,
                "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
                "train_roc_auc": train_roc_auc,
                "test_roc_auc": test_roc_auc,
                "auc_gap": auc_gap,
                "is_overfit": auc_gap >= 0.05,
            }
        )

    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        [["f1_score", "train_roc_auc", "test_roc_auc", "auc_gap", "is_overfit"]]
        .round(4)
        .sort_values(
            by=["is_overfit", "test_roc_auc"],
            ascending=[True, False],
        )
    )

    print(results_df.to_string())

분석 기준: is_promotion == 1
양성 클래스 기준: is_repurchase == 0
타깃 분포
                   count  percent
is_repurchase_num                
0                   8060    67.42
1                   3895    32.58
train/test 분리 비율: train 80.0% / test 20.0%
train 데이터 수: 9564
test 데이터 수: 2391
                    f1_score  train_roc_auc  test_roc_auc  auc_gap  is_overfit
model                                                                         
CatBoost              0.4273         0.5858        0.5566   0.0293       False
GradientBoost         0.0025         0.5772        0.5535   0.0237       False
XGBoost               0.4191         0.5931        0.5520   0.0411       False
LogisticRegression    0.4373         0.5446        0.5516  -0.0070       False
SVM                   0.4509         0.5676        0.5468   0.0208       False
RandomForest          0.4338         0.6115        0.5502   0.0612        True
LightGBM              0.4292         0.6069        0.5477   0.0593        True


# 1차 파생 변수 (19개)

## 전체

In [15]:
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# LightGBM 경고 숨김부
warnings.filterwarnings(
    "ignore",
    message=".*X does not have valid feature names.*LGBMClassifier.*",
    category=UserWarning,
)

# 결과 출력 형식 설정부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_최종파생변수(80개).csv"

# 사용 변수 설정부
selected_features = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "is_basic",
    "is_premium",
    "age_group",
    "is_female",
    "payment_is_mobile",
    "payment_is_pc",
    "payment_is_android",
    "reg_is_weekend",
    "reg_hour_morning",
    "reg_hour_evening",
    "reg_hour_night",
    "movie_per_active_day",
    "max_day_share",
    "weekend_watch_ratio",
    "watch_ratio_under_1m",
    "day_count_over_3times",
    "new_movie_in_90d_ratio",
    "old_movie_ratio(5y)",
    "avg_ott_release_year",
    "genre_diversity_count",
]

# 사용 컬럼 설정부
use_cols = [
    *selected_features,
    "is_repurchase",
]


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 타깃 분포 출력 함수부
def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


# 전처리 파이프라인 생성 함수부
def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return preprocessor


# 모델 생성 함수부
def make_models(scale_pos_weight):
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "SVM": SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoost": GradientBoostingClassifier(
            random_state=42,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=4,
            auto_class_weights="Balanced",
            random_state=42,
            verbose=0,
            allow_writing_files=False,
        ),
    }

    return models


# 데이터 로드부
df = pd.read_csv(file_path, usecols=use_cols).copy()

# 숫자형 변수 설정부
numeric_features = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "is_basic",
    "is_premium",
    "is_female",
    "payment_is_mobile",
    "payment_is_pc",
    "payment_is_android",
    "reg_is_weekend",
    "reg_hour_morning",
    "reg_hour_evening",
    "reg_hour_night",
    "movie_per_active_day",
    "max_day_share",
    "weekend_watch_ratio",
    "watch_ratio_under_1m",
    "day_count_over_3times",
    "new_movie_in_90d_ratio",
    "old_movie_ratio(5y)",
    "avg_ott_release_year",
    "genre_diversity_count",
]

# 범주형 변수 설정부
categorical_features = [
    "age_group",
]

# 숫자형 변환부
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "is_basic",
    "is_premium",
    "is_female",
    "payment_is_mobile",
    "payment_is_pc",
    "payment_is_android",
    "reg_is_weekend",
    "reg_hour_morning",
    "reg_hour_evening",
    "reg_hour_night",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[selected_features].copy()

# 양성 클래스 정의부
y = (df["is_repurchase_num"] == 0).astype(int)

print("분석 기준: 전체 데이터 + 1차 파생 변수 조합")
print("양성 클래스 기준: is_repurchase == 0")
print(f"전체 데이터 수: {len(df)}")
print(f"사용 변수 수: {len(selected_features)}")
print("사용 변수")
print(selected_features)
print("타깃 분포")
print_target_distribution(y)

# 학습 가능 여부 확인부
if y.nunique() < 2 or y.value_counts().min() < 2:
    print("학습 불가: 타깃 클래스가 부족합니다.")
else:
    # 학습/평가 데이터 분리부
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    print(f"train/test 분리 비율: train {len(X_train) / len(X) * 100:.1f}% / test {len(X_test) / len(X) * 100:.1f}%")
    print(f"train 데이터 수: {len(X_train)}")
    print(f"test 데이터 수: {len(X_test)}")

    # XGBoost 불균형 가중치 계산부
    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    # 모델 정의부
    models = make_models(scale_pos_weight)

    # 평가 수행부
    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        clf.fit(X_train, y_train)

        y_test_pred = clf.predict(X_test)

        y_train_proba = clf.predict_proba(X_train)[:, 1]
        y_test_proba = clf.predict_proba(X_test)[:, 1]

        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_test_proba)
        auc_gap = train_roc_auc - test_roc_auc

        result = {
            "model": model_name,
            "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
            "train_roc_auc": train_roc_auc,
            "test_roc_auc": test_roc_auc,
            "auc_gap": auc_gap,
            "is_overfit": auc_gap >= 0.05,
        }

        results.append(result)

    # 결과 출력부
    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        [
            [
                "f1_score",
                "train_roc_auc",
                "test_roc_auc",
                "auc_gap",
                "is_overfit",
            ]
        ]
        .round(4)
        .sort_values(
            by=["is_overfit", "test_roc_auc"],
            ascending=[True, False],
        )
    )

    print(results_df.to_string())

분석 기준: 전체 데이터 + 1차 파생 변수 조합
양성 클래스 기준: is_repurchase == 0
전체 데이터 수: 23081
사용 변수 수: 23
사용 변수
['is_promotion', 'is_churn_prevented', 'is_user_verified', 'is_basic', 'is_premium', 'age_group', 'is_female', 'payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'reg_is_weekend', 'reg_hour_morning', 'reg_hour_evening', 'reg_hour_night', 'movie_per_active_day', 'max_day_share', 'weekend_watch_ratio', 'watch_ratio_under_1m', 'day_count_over_3times', 'new_movie_in_90d_ratio', 'old_movie_ratio(5y)', 'avg_ott_release_year', 'genre_diversity_count']
타깃 분포
                   count  percent
is_repurchase_num                
0                  16557    71.73
1                   6524    28.27
train/test 분리 비율: train 80.0% / test 20.0%
train 데이터 수: 18464
test 데이터 수: 4617
                    f1_score  train_roc_auc  test_roc_auc  auc_gap  is_overfit
model                                                                         
CatBoost              0.5086         0.7409        0.6992   0.0417     

## 비프로모션

In [ ]:
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# LightGBM 경고 숨김부
warnings.filterwarnings(
    "ignore",
    message=".*X does not have valid feature names.*LGBMClassifier.*",
    category=UserWarning,
)

# 결과 출력 형식 설정부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_최종파생변수(80개).csv"

# 모델 입력 변수 설정부
selected_features = [
    "is_churn_prevented",
    "is_user_verified",
    "is_basic",
    "is_premium",
    "age_group",
    "is_female",
    "payment_is_mobile",
    "payment_is_pc",
    "payment_is_android",
    "reg_is_weekend",
    "reg_hour_morning",
    "reg_hour_evening",
    "reg_hour_night",
    "movie_per_active_day",
    "max_day_share",
    "weekend_watch_ratio",
    "watch_ratio_under_1m",
    "day_count_over_3times",
    "new_movie_in_90d_ratio",
    "old_movie_ratio(5y)",
    "avg_ott_release_year",
    "genre_diversity_count",
]

# 사용 컬럼 설정부
use_cols = [
    "is_promotion",
    *selected_features,
    "is_repurchase",
]


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 타깃 분포 출력 함수부
def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


# 전처리 파이프라인 생성 함수부
def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return preprocessor


# 모델 생성 함수부
def make_models(scale_pos_weight):
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "SVM": SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoost": GradientBoostingClassifier(
            random_state=42,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=4,
            auto_class_weights="Balanced",
            random_state=42,
            verbose=0,
            allow_writing_files=False,
        ),
    }

    return models


# 데이터 로드부
df = pd.read_csv(file_path, usecols=use_cols).copy()

# 숫자형 변수 설정부
numeric_features = [
    "is_churn_prevented",
    "is_user_verified",
    "is_basic",
    "is_premium",
    "is_female",
    "payment_is_mobile",
    "payment_is_pc",
    "payment_is_android",
    "reg_is_weekend",
    "reg_hour_morning",
    "reg_hour_evening",
    "reg_hour_night",
    "movie_per_active_day",
    "max_day_share",
    "weekend_watch_ratio",
    "watch_ratio_under_1m",
    "day_count_over_3times",
    "new_movie_in_90d_ratio",
    "old_movie_ratio(5y)",
    "avg_ott_release_year",
    "genre_diversity_count",
]

# 범주형 변수 설정부
categorical_features = [
    "age_group",
]

# 숫자형 변환부
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "is_basic",
    "is_premium",
    "is_female",
    "payment_is_mobile",
    "payment_is_pc",
    "payment_is_android",
    "reg_is_weekend",
    "reg_hour_morning",
    "reg_hour_evening",
    "reg_hour_night",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 비프로모션 데이터 분리부
df_promotion_0 = df[df["is_promotion"] == 0].copy()

# 입력 변수, 타깃 변수 생성부
X = df_promotion_0[selected_features].copy()
y = (df_promotion_0["is_repurchase_num"] == 0).astype(int)

print("분석 기준: is_promotion == 0 + 1차 파생 변수 조합")
print("양성 클래스 기준: is_repurchase == 0")
print(f"전체 데이터 수: {len(df_promotion_0)}")
print(f"사용 변수 수: {len(selected_features)}")
print("타깃 분포")
print_target_distribution(y)

# 학습 가능 여부 확인부
if y.nunique() < 2 or y.value_counts().min() < 2:
    print("학습 불가: 타깃 클래스가 부족합니다.")
else:
    # 학습/평가 데이터 분리부
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    print(f"train/test 분리 비율: train {len(X_train) / len(X) * 100:.1f}% / test {len(X_test) / len(X) * 100:.1f}%")
    print(f"train 데이터 수: {len(X_train)}")
    print(f"test 데이터 수: {len(X_test)}")

    # XGBoost 불균형 가중치 계산부
    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    # 모델 정의부
    models = make_models(scale_pos_weight)

    # 평가 수행부
    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        clf.fit(X_train, y_train)

        y_test_pred = clf.predict(X_test)

        y_train_proba = clf.predict_proba(X_train)[:, 1]
        y_test_proba = clf.predict_proba(X_test)[:, 1]

        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_test_proba)
        auc_gap = train_roc_auc - test_roc_auc

        result = {
            "model": model_name,
            "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
            "train_roc_auc": train_roc_auc,
            "test_roc_auc": test_roc_auc,
            "auc_gap": auc_gap,
            "is_overfit": auc_gap >= 0.05,
        }

        results.append(result)

    # 결과 출력부
    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        [
            [
                "f1_score",
                "train_roc_auc",
                "test_roc_auc",
                "auc_gap",
                "is_overfit",
            ]
        ]
        .round(4)
        .sort_values(
            by=["is_overfit", "test_roc_auc"],
            ascending=[True, False],
        )
    )

    print(results_df.to_string())

분석 기준: is_promotion == 0 + 1차 파생 변수 조합
양성 클래스 기준: is_repurchase == 0
전체 데이터 수: 11177
사용 변수 수: 22
사용 변수
['is_churn_prevented', 'is_user_verified', 'is_basic', 'is_premium', 'age_group', 'is_female', 'payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'reg_is_weekend', 'reg_hour_morning', 'reg_hour_evening', 'reg_hour_night', 'movie_per_active_day', 'max_day_share', 'weekend_watch_ratio', 'watch_ratio_under_1m', 'day_count_over_3times', 'new_movie_in_90d_ratio', 'old_movie_ratio(5y)', 'avg_ott_release_year', 'genre_diversity_count']
타깃 분포
                   count  percent
is_repurchase_num                
0                   8520    76.23
1                   2657    23.77
train/test 분리 비율: train 80.0% / test 20.0%
train 데이터 수: 8941
test 데이터 수: 2236
                    f1_score  train_roc_auc  test_roc_auc  auc_gap  is_overfit
model                                                                         
LogisticRegression    0.4674         0.7044        0.7048  -0.0004       Fals

## 프로모션

In [18]:
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# LightGBM 경고 숨김부
warnings.filterwarnings(
    "ignore",
    message=".*X does not have valid feature names.*LGBMClassifier.*",
    category=UserWarning,
)

# 결과 출력 형식 설정부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_최종파생변수(80개).csv"

# 모델 입력 변수 설정부
selected_features = [
    "is_churn_prevented",
    "is_user_verified",
    "is_basic",
    "is_premium",
    "age_group",
    "is_female",
    "payment_is_mobile",
    "payment_is_pc",
    "payment_is_android",
    "reg_is_weekend",
    "reg_hour_morning",
    "reg_hour_evening",
    "reg_hour_night",
    "movie_per_active_day",
    "max_day_share",
    "weekend_watch_ratio",
    "watch_ratio_under_1m",
    "day_count_over_3times",
    "new_movie_in_90d_ratio",
    "old_movie_ratio(5y)",
    "avg_ott_release_year",
    "genre_diversity_count",
]

# 사용 컬럼 설정부
use_cols = [
    "is_promotion",
    *selected_features,
    "is_repurchase",
]


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 타깃 분포 출력 함수부
def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


# 전처리 파이프라인 생성 함수부
def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return preprocessor


# 모델 생성 함수부
def make_models(scale_pos_weight):
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "SVM": SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoost": GradientBoostingClassifier(
            random_state=42,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=4,
            auto_class_weights="Balanced",
            random_state=42,
            verbose=0,
            allow_writing_files=False,
        ),
    }

    return models


# 데이터 로드부
df = pd.read_csv(file_path, usecols=use_cols).copy()

# 숫자형 변수 설정부
numeric_features = [
    "is_churn_prevented",
    "is_user_verified",
    "is_basic",
    "is_premium",
    "is_female",
    "payment_is_mobile",
    "payment_is_pc",
    "payment_is_android",
    "reg_is_weekend",
    "reg_hour_morning",
    "reg_hour_evening",
    "reg_hour_night",
    "movie_per_active_day",
    "max_day_share",
    "weekend_watch_ratio",
    "watch_ratio_under_1m",
    "day_count_over_3times",
    "new_movie_in_90d_ratio",
    "old_movie_ratio(5y)",
    "avg_ott_release_year",
    "genre_diversity_count",
]

# 범주형 변수 설정부
categorical_features = [
    "age_group",
]

# 숫자형 변환부
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "is_basic",
    "is_premium",
    "is_female",
    "payment_is_mobile",
    "payment_is_pc",
    "payment_is_android",
    "reg_is_weekend",
    "reg_hour_morning",
    "reg_hour_evening",
    "reg_hour_night",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 프로모션 데이터 분리부
df_promotion_1 = df[df["is_promotion"] == 1].copy()

# 입력 변수, 타깃 변수 생성부
X = df_promotion_1[selected_features].copy()
y = (df_promotion_1["is_repurchase_num"] == 0).astype(int)

print("분석 기준: is_promotion == 1 + 1차 파생 변수 조합")
print("양성 클래스 기준: is_repurchase == 0")
print(f"전체 데이터 수: {len(df_promotion_1)}")
print(f"사용 변수 수: {len(selected_features)}")
print("사용 변수")
print(selected_features)
print("타깃 분포")
print_target_distribution(y)

# 학습 가능 여부 확인부
if y.nunique() < 2 or y.value_counts().min() < 2:
    print("학습 불가: 타깃 클래스가 부족합니다.")
else:
    # 학습/평가 데이터 분리부
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    print(f"train/test 분리 비율: train {len(X_train) / len(X) * 100:.1f}% / test {len(X_test) / len(X) * 100:.1f}%")
    print(f"train 데이터 수: {len(X_train)}")
    print(f"test 데이터 수: {len(X_test)}")

    # XGBoost 불균형 가중치 계산부
    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    # 모델 정의부
    models = make_models(scale_pos_weight)

    # 평가 수행부
    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        clf.fit(X_train, y_train)

        y_test_pred = clf.predict(X_test)

        y_train_proba = clf.predict_proba(X_train)[:, 1]
        y_test_proba = clf.predict_proba(X_test)[:, 1]

        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_test_proba)
        auc_gap = train_roc_auc - test_roc_auc

        result = {
            "model": model_name,
            "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
            "train_roc_auc": train_roc_auc,
            "test_roc_auc": test_roc_auc,
            "auc_gap": auc_gap,
            "is_overfit": auc_gap >= 0.05,
        }

        results.append(result)

    # 결과 출력부
    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        [
            [
                "f1_score",
                "train_roc_auc",
                "test_roc_auc",
                "auc_gap",
                "is_overfit",
            ]
        ]
        .round(4)
        .sort_values(
            by=["is_overfit", "test_roc_auc"],
            ascending=[True, False],
        )
    )

    print(results_df.to_string())

분석 기준: is_promotion == 1 + 1차 파생 변수 조합
양성 클래스 기준: is_repurchase == 0
전체 데이터 수: 11904
사용 변수 수: 22
사용 변수
['is_churn_prevented', 'is_user_verified', 'is_basic', 'is_premium', 'age_group', 'is_female', 'payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'reg_is_weekend', 'reg_hour_morning', 'reg_hour_evening', 'reg_hour_night', 'movie_per_active_day', 'max_day_share', 'weekend_watch_ratio', 'watch_ratio_under_1m', 'day_count_over_3times', 'new_movie_in_90d_ratio', 'old_movie_ratio(5y)', 'avg_ott_release_year', 'genre_diversity_count']
타깃 분포
                   count  percent
is_repurchase_num                
0                   8037    67.52
1                   3867    32.48
train/test 분리 비율: train 80.0% / test 20.0%
train 데이터 수: 9523
test 데이터 수: 2381
                    f1_score  train_roc_auc  test_roc_auc  auc_gap  is_overfit
model                                                                         
LogisticRegression    0.5189         0.6812        0.6740   0.0072       Fals

# 최종 파생 변수

## 전체

In [22]:
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# LightGBM 경고 숨김부
warnings.filterwarnings(
    "ignore",
    message=".*X does not have valid feature names.*LGBMClassifier.*",
    category=UserWarning,
)

# 결과 출력 형식 설정부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_최종파생변수(80개).csv"

# 모델 입력 제외 컬럼 설정부
exclude_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "payment_device",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
]


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 타깃 분포 출력 함수부
def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


# 전처리 파이프라인 생성 함수부
def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return preprocessor


# 모델 생성 함수부
def make_models(scale_pos_weight):
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "SVM": SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoost": GradientBoostingClassifier(
            random_state=42,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=4,
            auto_class_weights="Balanced",
            random_state=42,
            verbose=0,
            allow_writing_files=False,
        ),
    }

    return models


# 데이터 로드부
df = pd.read_csv(file_path).copy()

# 전체 컬럼 기준 입력 변수 생성부
selected_features = [
    col for col in df.columns if col not in exclude_cols
]

# 범주형 변수 설정부
categorical_features = [
    "age_group",
]

# 숫자형 변수 설정부
numeric_features = [
    col for col in selected_features if col not in categorical_features
]

# 숫자형 변환부
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 타깃 변수 변환부
df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[selected_features].copy()

# 양성 클래스 정의부
y = (df["is_repurchase_num"] == 0).astype(int)

print("분석 기준: 전체 데이터 + 최종 파생 변수")
print("양성 클래스 기준: is_repurchase == 0")
print(f"전체 데이터 수: {len(df)}")
print(f"전체 컬럼 수: {len(pd.read_csv(file_path, nrows=0).columns)}")
print(f"모델 입력 제외 컬럼 수: {len(exclude_cols)}")
print(f"사용 변수 수: {len(selected_features)}")
print("사용 변수")
print(selected_features)
print("타깃 분포")
print_target_distribution(y)

# 학습 가능 여부 확인부
if y.nunique() < 2 or y.value_counts().min() < 2:
    print("학습 불가: 타깃 클래스가 부족합니다.")
else:
    # 학습/평가 데이터 분리부
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    print(f"train/test 분리 비율: train {len(X_train) / len(X) * 100:.1f}% / test {len(X_test) / len(X) * 100:.1f}%")
    print(f"train 데이터 수: {len(X_train)}")
    print(f"test 데이터 수: {len(X_test)}")

    # XGBoost 불균형 가중치 계산부
    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    # 모델 정의부
    models = make_models(scale_pos_weight)

    # 평가 수행부
    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        clf.fit(X_train, y_train)

        y_test_pred = clf.predict(X_test)

        y_train_proba = clf.predict_proba(X_train)[:, 1]
        y_test_proba = clf.predict_proba(X_test)[:, 1]

        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_test_proba)
        auc_gap = train_roc_auc - test_roc_auc

        result = {
            "model": model_name,
            "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
            "train_roc_auc": train_roc_auc,
            "test_roc_auc": test_roc_auc,
            "auc_gap": auc_gap,
            "is_overfit": auc_gap >= 0.05,
        }

        results.append(result)

    # 결과 출력부
    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        [
            [
                "f1_score",
                "train_roc_auc",
                "test_roc_auc",
                "auc_gap",
                "is_overfit",
            ]
        ]
        .round(4)
        .sort_values(
            by=["is_overfit", "test_roc_auc"],
            ascending=[True, False],
        )
    )

    print(results_df.to_string())

분석 기준: 전체 데이터 + 최종 파생 변수
양성 클래스 기준: is_repurchase == 0
전체 데이터 수: 23081
전체 컬럼 수: 91
모델 입력 제외 컬럼 수: 11
사용 변수 수: 80
사용 변수
['is_promotion', 'is_churn_prevented', 'is_user_verified', 'is_basic', 'is_standard', 'is_premium', 'age_group', 'is_female', 'is_male', 'payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'payment_is_ios', 'reg_is_weekend', 'reg_hour_morning', 'reg_hour_afternoon', 'reg_hour_evening', 'reg_hour_night', 'total_watch_count', 'unique_movie', 'watch_days', 'total_watch_time(min)', 'active_ratio', 'watch_per_day', 'avg_watch_time(min)', 'median_watch_time(min)', 'std_watch_time(min)', 'max_watch_time(min)', 'avg_daily_watch_time(min)', 'max_daily_watch_time(min)', 'max_daily_sessions', 'recency', 'avg_gap_between_watch_days', 'max_inactive_gap_days', 'avg_gap_w1_watch_days', 'avg_gap_w2_watch_days', 'avg_gap_w3_watch_days', 'avg_rewatch_ratio', 'weekend_watch_ratio', 'watch_ratio_under_1m', 'watch_ratio_under_5m', 'is_cold_start_3d', 'is_cold_start_7d', 'movie_per_

## 비프로모션

In [23]:
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# LightGBM 경고 숨김부
warnings.filterwarnings(
    "ignore",
    message=".*X does not have valid feature names.*LGBMClassifier.*",
    category=UserWarning,
)

# 결과 출력 형식 설정부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_최종파생변수(80개).csv"

# 모델 입력 제외 컬럼 설정부
exclude_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "payment_device",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
    "is_promotion",
]


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 타깃 분포 출력 함수부
def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


# 전처리 파이프라인 생성 함수부
def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return preprocessor


# 모델 생성 함수부
def make_models(scale_pos_weight):
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "SVM": SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoost": GradientBoostingClassifier(
            random_state=42,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=4,
            auto_class_weights="Balanced",
            random_state=42,
            verbose=0,
            allow_writing_files=False,
        ),
    }

    return models


# 데이터 로드부
df = pd.read_csv(file_path).copy()

# is_promotion 변환부
df["is_promotion"] = to_binary(df["is_promotion"])

# 비프로모션 데이터 분리부
df_promotion_0 = df[df["is_promotion"] == 0].copy()

# 전체 컬럼 기준 입력 변수 생성부
selected_features = [
    col for col in df_promotion_0.columns if col not in exclude_cols
]

# 범주형 변수 설정부
categorical_features = [
    "age_group",
]

# 숫자형 변수 설정부
numeric_features = [
    col for col in selected_features if col not in categorical_features
]

# 숫자형 변환부
for col in numeric_features:
    df_promotion_0[col] = pd.to_numeric(df_promotion_0[col], errors="coerce")

# 타깃 변수 변환부
df_promotion_0["is_repurchase_num"] = to_binary(df_promotion_0["is_repurchase"])

# 타깃 결측 제거부
df_promotion_0 = df_promotion_0[df_promotion_0["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df_promotion_0[selected_features].copy()

# 양성 클래스 정의부
y = (df_promotion_0["is_repurchase_num"] == 0).astype(int)

print("분석 기준: is_promotion == 0 + 최종 파생 변수")
print("양성 클래스 기준: is_repurchase == 0")
print(f"전체 데이터 수: {len(df_promotion_0)}")
print(f"전체 컬럼 수: {len(pd.read_csv(file_path, nrows=0).columns)}")
print(f"모델 입력 제외 컬럼 수: {len(exclude_cols)}")
print(f"사용 변수 수: {len(selected_features)}")
print("타깃 분포")
print_target_distribution(y)

# 학습 가능 여부 확인부
if y.nunique() < 2 or y.value_counts().min() < 2:
    print("학습 불가: 타깃 클래스가 부족합니다.")
else:
    # 학습/평가 데이터 분리부
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    print(f"train/test 분리 비율: train {len(X_train) / len(X) * 100:.1f}% / test {len(X_test) / len(X) * 100:.1f}%")
    print(f"train 데이터 수: {len(X_train)}")
    print(f"test 데이터 수: {len(X_test)}")

    # XGBoost 불균형 가중치 계산부
    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    # 모델 정의부
    models = make_models(scale_pos_weight)

    # 평가 수행부
    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        clf.fit(X_train, y_train)

        y_test_pred = clf.predict(X_test)

        y_train_proba = clf.predict_proba(X_train)[:, 1]
        y_test_proba = clf.predict_proba(X_test)[:, 1]

        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_test_proba)
        auc_gap = train_roc_auc - test_roc_auc

        result = {
            "model": model_name,
            "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
            "train_roc_auc": train_roc_auc,
            "test_roc_auc": test_roc_auc,
            "auc_gap": auc_gap,
            "is_overfit": auc_gap >= 0.05,
        }

        results.append(result)

    # 결과 출력부
    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        [
            [
                "f1_score",
                "train_roc_auc",
                "test_roc_auc",
                "auc_gap",
                "is_overfit",
            ]
        ]
        .round(4)
        .sort_values(
            by=["is_overfit", "test_roc_auc"],
            ascending=[True, False],
        )
    )

    print(results_df.to_string())

분석 기준: is_promotion == 0 + 최종 파생 변수
양성 클래스 기준: is_repurchase == 0
전체 데이터 수: 11177
전체 컬럼 수: 91
모델 입력 제외 컬럼 수: 12
사용 변수 수: 79
타깃 분포
                   count  percent
is_repurchase_num                
0                   8520    76.23
1                   2657    23.77
train/test 분리 비율: train 80.0% / test 20.0%
train 데이터 수: 8941
test 데이터 수: 2236
                    f1_score  train_roc_auc  test_roc_auc  auc_gap  is_overfit
model                                                                         
CatBoost              0.6565         0.9184        0.8871   0.0313       False
GradientBoost         0.5960         0.9132        0.8845   0.0287       False
LogisticRegression    0.6154         0.8705        0.8577   0.0128       False
XGBoost               0.6662         0.9496        0.8862   0.0633        True
LightGBM              0.6628         0.9863        0.8794   0.1068        True
RandomForest          0.5982         0.9961        0.8632   0.1329        True
SVM                   0.

## 프로모션

In [21]:
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# LightGBM 경고 숨김부
warnings.filterwarnings(
    "ignore",
    message=".*X does not have valid feature names.*LGBMClassifier.*",
    category=UserWarning,
)

# 결과 출력 형식 설정부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_최종파생변수(80개).csv"

# 모델 입력 제외 컬럼 설정부
exclude_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "payment_device",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
    "is_promotion",
]


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 타깃 분포 출력 함수부
def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )

    print(target_dist)


# 전처리 파이프라인 생성 함수부
def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return preprocessor


# 모델 생성 함수부
def make_models(scale_pos_weight):
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "SVM": SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoost": GradientBoostingClassifier(
            random_state=42,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=4,
            auto_class_weights="Balanced",
            random_state=42,
            verbose=0,
            allow_writing_files=False,
        ),
    }

    return models


# 데이터 로드부
df = pd.read_csv(file_path).copy()

# is_promotion 변환부
df["is_promotion"] = to_binary(df["is_promotion"])

# 프로모션 데이터 분리부
df_promotion_1 = df[df["is_promotion"] == 1].copy()

# 전체 컬럼 기준 입력 변수 생성부
selected_features = [
    col for col in df_promotion_1.columns if col not in exclude_cols
]

# 범주형 변수 설정부
categorical_features = [
    "age_group",
]

# 숫자형 변수 설정부
numeric_features = [
    col for col in selected_features if col not in categorical_features
]

# 숫자형 변환부
for col in numeric_features:
    df_promotion_1[col] = pd.to_numeric(df_promotion_1[col], errors="coerce")

# 타깃 변수 변환부
df_promotion_1["is_repurchase_num"] = to_binary(df_promotion_1["is_repurchase"])

# 타깃 결측 제거부
df_promotion_1 = df_promotion_1[df_promotion_1["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df_promotion_1[selected_features].copy()

# 양성 클래스 정의부
y = (df_promotion_1["is_repurchase_num"] == 0).astype(int)

print("분석 기준: is_promotion == 1 + 최종 파생 변수")
print("양성 클래스 기준: is_repurchase == 0")
print(f"전체 데이터 수: {len(df_promotion_1)}")
print(f"전체 컬럼 수: {len(pd.read_csv(file_path, nrows=0).columns)}")
print(f"모델 입력 제외 컬럼 수: {len(exclude_cols)}")
print(f"사용 변수 수: {len(selected_features)}")
print("타깃 분포")
print_target_distribution(y)

# 학습 가능 여부 확인부
if y.nunique() < 2 or y.value_counts().min() < 2:
    print("학습 불가: 타깃 클래스가 부족합니다.")
else:
    # 학습/평가 데이터 분리부
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    print(f"train/test 분리 비율: train {len(X_train) / len(X) * 100:.1f}% / test {len(X_test) / len(X) * 100:.1f}%")
    print(f"train 데이터 수: {len(X_train)}")
    print(f"test 데이터 수: {len(X_test)}")

    # XGBoost 불균형 가중치 계산부
    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    # 모델 정의부
    models = make_models(scale_pos_weight)

    # 평가 수행부
    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
                ("model", model),
            ]
        )

        clf.fit(X_train, y_train)

        y_test_pred = clf.predict(X_test)

        y_train_proba = clf.predict_proba(X_train)[:, 1]
        y_test_proba = clf.predict_proba(X_test)[:, 1]

        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_test_proba)
        auc_gap = train_roc_auc - test_roc_auc

        result = {
            "model": model_name,
            "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
            "train_roc_auc": train_roc_auc,
            "test_roc_auc": test_roc_auc,
            "auc_gap": auc_gap,
            "is_overfit": auc_gap >= 0.05,
        }

        results.append(result)

    # 결과 출력부
    results_df = (
        pd.DataFrame(results)
        .set_index("model")
        [
            [
                "f1_score",
                "train_roc_auc",
                "test_roc_auc",
                "auc_gap",
                "is_overfit",
            ]
        ]
        .round(4)
        .sort_values(
            by=["is_overfit", "test_roc_auc"],
            ascending=[True, False],
        )
    )

    print(results_df.to_string())

분석 기준: is_promotion == 1 + 최종 파생 변수
양성 클래스 기준: is_repurchase == 0
전체 데이터 수: 11904
전체 컬럼 수: 91
모델 입력 제외 컬럼 수: 12
사용 변수 수: 79
타깃 분포
                   count  percent
is_repurchase_num                
0                   8037    67.52
1                   3867    32.48
train/test 분리 비율: train 80.0% / test 20.0%
train 데이터 수: 9523
test 데이터 수: 2381
                    f1_score  train_roc_auc  test_roc_auc  auc_gap  is_overfit
model                                                                         
CatBoost              0.6910         0.8951        0.8547   0.0404       False
GradientBoost         0.6151         0.8934        0.8504   0.0430       False
LogisticRegression    0.6607         0.8461        0.8277   0.0184       False
XGBoost               0.6935         0.9318        0.8552   0.0767        True
LightGBM              0.6895         0.9793        0.8493   0.1300        True
RandomForest          0.6342         0.9968        0.8348   0.1619        True
SVM                   0.